# AML Benchmark — Part B PAI-HNU Run Notebook

**Part B primary strategy:** Part-A-Informed Hard-Negative Undersampling (**PAI-HNU**)  
**Goal:** Level-3 Mini-End-to-End-Smoke first, then full Part-B runs only after manual confirmation.

This notebook is intentionally staged:

1. Mount Drive and clone the Part-B branch.
2. Install dependencies and verify the sampler tests.
3. Copy/reuse Part-A artefacts from Drive.
4. Generate/cache baseline train scores.
5. Run **Mini-End-to-End-Smoke** with `--sample-n-train`.
6. Inspect smoke outputs.
7. Only after manual confirmation: run all three full PAI-HNU prevalences.
8. Generate Table 6 and back up results to Drive.

**Do not run the Full-Run cells until the smoke outputs are checked.**


## 0 — Configuration


In [ ]:
from pathlib import Path
import os, json, shutil, datetime, subprocess, sys

# === Edit only if your paths/branch changed ===
GITHUB_REPO = "https://github.com/fdrmic/classimbalance.git"
BRANCH = "feature/part-b-hard-negative-undersampling"

PROJECT_DIR = Path("/content/classimbalance")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_RESULTS = DRIVE_ROOT / "aml_results"

# Existing Part-A completed run on Drive
PART_A_RUN_DIR = DRIVE_RESULTS / "large_run_v2_20260407_1904"

# Existing Part-A XGBoost Baseline model used to score train rows for hard-negative mining
BASELINE_MODEL_PATH = PART_A_RUN_DIR / "runs" / "xgboost__baseline__p001__20260404_143052" / "model.pkl"

# PAI-HNU config inside the repo
PATHS_CONFIG = PROJECT_DIR / "configs" / "paths_large_part_b_pai_hnu.yaml"

# Smoke size. 200k is large enough to catch alignment/I/O issues but still cheap.
SMOKE_N_TRAIN = 200_000
SMOKE_PREVALENCE = "0.01"

print("PART_A_RUN_DIR      :", PART_A_RUN_DIR)
print("BASELINE_MODEL_PATH :", BASELINE_MODEL_PATH)
print("PROJECT_DIR         :", PROJECT_DIR)


PART_A_RUN_DIR      : /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904
BASELINE_MODEL_PATH : /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl
PROJECT_DIR         : /content/classimbalance


## 1 — Mount Google Drive


In [2]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")


Mounted at /content/drive
Drive mounted.


## 2 — Runtime check: RAM and GPU


In [ ]:
import psutil, os, subprocess, textwrap

ram = psutil.virtual_memory()
print(f"Total RAM     : {ram.total / 1e9:.1f} GB")
print(f"Available RAM : {ram.available / 1e9:.1f} GB")
print()

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "No GPU detected"

if ram.total < 80e9:
    print("\nWARNING: Less than ~80 GB RAM detected. For full runs, use Colab Pro/Pro+ high-RAM runtime.")


Total RAM     : 179.4 GB
Available RAM : 176.8 GB

NVIDIA A100-SXM4-80GB, 81920 MiB, 81153 MiB


## 3 — Clone repository branch


In [ ]:
import os, shutil
from pathlib import Path

if PROJECT_DIR.exists():
    print(f"Removing existing clone: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

cmd = ["git", "clone", "-b", BRANCH, GITHUB_REPO, str(PROJECT_DIR)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Latest commit:")
print(subprocess.check_output(["git", "--no-pager", "log", "--oneline", "-1"], text=True))


Running: git clone -b feature/part-b-hard-negative-undersampling https://github.com/fdrmic/classimbalance.git /content/classimbalance
Working directory: /content/classimbalance
Branch: feature/part-b-hard-negative-undersampling
Latest commit:
de553a1 startegie PAI-HNU



## 4 — Install dependencies


In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_DIR)

# PyYAML is needed for config editing; pytest for Level-1 tests.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest"], check=True)

# Project install
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Requirements, if present
req = PROJECT_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

print("Installation complete.")


Installation complete.


In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = Path("/content/classimbalance")
os.chdir(PROJECT_DIR)

print("PWD:", os.getcwd())
print("Repo files:", [p.name for p in PROJECT_DIR.iterdir()][:20])

PWD: /content/classimbalance
Repo files: ['results', 'aml_part_b_multi_threshold_run.ipynb', 'src', 'tests', 'requirements.txt', 'configs', 'notebooks', '0', 'scripts', 'test_pipeline.py', 'pyproject.toml', 'aml_large_run.ipynb', 'README_PROJECT_STRUCTURE.md', 'docs', 'analysis', '.git', '.gitignore']


In [ ]:
!git status
!git branch --show-current
!ls src/aml_benchmark/sampling/

On branch feature/part-b-hard-negative-undersampling
Your branch is up to date with 'origin/feature/part-b-hard-negative-undersampling'.

nothing to commit, working tree clean
feature/part-b-hard-negative-undersampling
hard_negative_undersampling.py	__init__.py  prevalence.py  strategies.py


In [ ]:
%cd /content/classimbalance
!python -m pip install -e .

/content
Obtaining file:///content/classimbalance
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for aml_benchmark (pyproject.toml) ... done
  Created wheel for aml_benchmark: filename=aml_benchmark-0.1.0-0.editable-py3-none-any.whl size=1410 sha256=71f053b6b6346ab065b5c5d2357e2ddbb781001742c3d0b5445affa8e8ed3dbd
  Stored in directory: /tmp/pip-ephem-wheel-cache-1tc15kec/wheels/c7/6d/93/c1343baf19d0f5af29e0132d7120f8e136a17356eb4b5e7274
Successfully built aml_benchmark
  Attempting uninstall: aml_benchmark
    Found existing installation: aml_benchmark 0.1.0
    Uninstalling aml_benchmark-0.1.0:
      Successfully uninstalled aml_benchmark-0.1.0


In [ ]:
import sys
sys.path.insert(0, "/content/classimbalance/src")

from aml_benchmark.sampling.hard_negative_undersampling import (
    build_pai_hnu_training_indices,
    validate_no_overlap,
)

print("Import OK")

Import OK


## 5 — Verify imports and run Level-1 unit tests


In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_DIR)

# Import check
from aml_benchmark.sampling.hard_negative_undersampling import build_pai_hnu_training_indices, validate_no_overlap
print("Import OK: PAI-HNU sampler available.")

# Level 1 tests
subprocess.run([sys.executable, "-m", "pytest", "tests/test_pai_hnu_sampler.py", "-v"], check=True)


Import OK: PAI-HNU sampler available.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', 'tests/test_pai_hnu_sampler.py', '-v'], returncode=0)

## 6 — Level-2 sampler smoke test


In [ ]:
import numpy as np
from aml_benchmark.sampling.hard_negative_undersampling import build_pai_hnu_training_indices, validate_no_overlap

rng = np.random.default_rng(0)
n = 100_000
n_pos = 120

y = np.zeros(n, dtype=np.int8)
y[rng.choice(n, size=n_pos, replace=False)] = 1
scores = rng.uniform(size=n).astype(np.float32)

sel = build_pai_hnu_training_indices(
    y,
    scores,
    target_prevalence=0.005,
    random_state=42,
)

validate_no_overlap(sel.pos_idx, sel.hard_neg_idx, sel.temporal_neg_idx, sel.global_neg_idx)
print("OK -", sel.counts)


2026-05-01 12:11:07 | INFO     | aml_benchmark.sampling.hard_negative_undersampling | Hard-negative cap binds: planned=11,940 capped=2,400 (cap = 20 * n_pos = 2,400). Reallocated 9,540 -> temporal+4,770, global+4,770.
2026-05-01 12:11:07 | INFO     | aml_benchmark.sampling.hard_negative_undersampling |   argpartition: n_neg=99,880 k=2,400 (~1.1 MB temp arrays)
2026-05-01 12:11:07 | INFO     | aml_benchmark.sampling.hard_negative_undersampling | PAI-HNU selection summary: {"n_pos": 120, "n_neg_total_available": 99880, "n_neg_target": 23880, "n_hard_planned": 11940, "n_hard_cap": 2400, "n_hard_actual": 2400, "n_temporal_planned": 10740, "n_temporal_actual": 10740, "n_global_planned": 10740, "n_global_actual": 10740, "n_total_neg_actual": 23880}
OK - {'n_pos': 120, 'n_neg_total_available': 99880, 'n_neg_target': 23880, 'n_hard_planned': 11940, 'n_hard_cap': 2400, 'n_hard_actual': 2400, 'n_temporal_planned': 10740, 'n_temporal_actual': 10740, 'n_global_planned': 10740, 'n_global_actual': 1

## 7 — Copy/reuse Part-A artefacts from Drive


In [ ]:
import shutil, os, json
from pathlib import Path

os.chdir(PROJECT_DIR)

if not PART_A_RUN_DIR.exists():
    raise FileNotFoundError(f"Part-A run folder not found: {PART_A_RUN_DIR}")

print("Using Part-A run folder:", PART_A_RUN_DIR)

# Destination folders expected by the project
splits_dst = PROJECT_DIR / "data" / "splits_v2"
processed_dst = PROJECT_DIR / "data" / "processed_v2"
runs_dst = PROJECT_DIR / "outputs" / "runs_v2"
leaderboard_dst = PROJECT_DIR / "outputs" / "leaderboard_v2"
results_dst = PROJECT_DIR / "results"

for p in [splits_dst, processed_dst, runs_dst, leaderboard_dst, results_dst]:
    p.mkdir(parents=True, exist_ok=True)

# Copy splits/features if present
for src_name, dst in [
    ("splits", splits_dst),
    ("splits_v2", splits_dst),
    ("processed", processed_dst),
    ("processed_v2", processed_dst),
    ("runs", runs_dst),
    ("runs_v2", runs_dst),
    ("leaderboard", leaderboard_dst),
    ("leaderboard_v2", leaderboard_dst),
]:
    src = PART_A_RUN_DIR / src_name
    if src.exists():
        print(f"Copying {src} -> {dst}")
        shutil.copytree(src, dst, dirs_exist_ok=True)

# Copy Part-A summary CSV if available in known locations.
summary_candidates = [
    PART_A_RUN_DIR / "part_a_summary_v2.csv",
    PART_A_RUN_DIR / "leaderboard" / "part_a_summary_v2.csv",
    PART_A_RUN_DIR / "leaderboard_v2" / "part_a_summary_v2.csv",
    PART_A_RUN_DIR / "outputs" / "leaderboard_v2" / "part_a_summary_v2.csv",
    DRIVE_RESULTS / "part_a_summary_v2.csv",
]
copied_summary = False
for cand in summary_candidates:
    if cand.exists():
        print("Copying Part-A summary:", cand)
        shutil.copy2(cand, results_dst / "part_a_summary_v2.csv")
        shutil.copy2(cand, leaderboard_dst / "part_a_summary_v2.csv")
        copied_summary = True
        break

if not copied_summary:
    print("WARNING: part_a_summary_v2.csv not found in known Drive locations.")
    print("Table 6 can be generated later after copying it to:")
    print(" -", results_dst / "part_a_summary_v2.csv")
    print(" -", leaderboard_dst / "part_a_summary_v2.csv")

print("\nSplits/features currently available:")
for f in sorted(splits_dst.glob("*"))[:40]:
    print(" -", f.relative_to(PROJECT_DIR), f.stat().st_size / 1e6, "MB")
print("...")

print("\nRuns available:")
for f in sorted(runs_dst.glob("xgboost__baseline__*"))[:10]:
    print(" -", f.relative_to(PROJECT_DIR))


Using Part-A run folder: /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904
Copying /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/splits -> /content/classimbalance/data/splits_v2
Copying /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/processed -> /content/classimbalance/data/processed_v2
Copying /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs -> /content/classimbalance/outputs/runs_v2
Copying /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/leaderboard -> /content/classimbalance/outputs/leaderboard_v2
Copying Part-A summary: /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/leaderboard/part_a_summary_v2.csv

Splits/features currently available:
 - data/splits_v2/feature_pipeline_v2.pkl 28.8928 MB
 - data/splits_v2/split_manifest.json 0.000836 MB
 - data/splits_v2/test.parquet 830.941973 MB
 - data/splits_v2/test_features_v2.parquet 1523.9732 MB
 - data/splits_v2/train.parquet 3859.471611 MB
 - data/sp

Zwischenschritt wurde alles korrekt gefunden

In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

print("PART_A_RUN_DIR:")
print(PART_A_RUN_DIR)
print("exists:", PART_A_RUN_DIR.exists())

print("\nBASELINE_MODEL_PATH:")
print(BASELINE_MODEL_PATH)
print("exists:", BASELINE_MODEL_PATH.exists())
if BASELINE_MODEL_PATH.exists():
    print("size MB:", round(BASELINE_MODEL_PATH.stat().st_size / 1e6, 2))

local_paths = {
    "split_manifest": PROJECT_DIR / "data" / "splits_v2" / "split_manifest.json",
    "train": PROJECT_DIR / "data" / "splits_v2" / "train.parquet",
    "val": PROJECT_DIR / "data" / "splits_v2" / "val.parquet",
    "test": PROJECT_DIR / "data" / "splits_v2" / "test.parquet",
    "train_features": PROJECT_DIR / "data" / "splits_v2" / "train_features_v2.parquet",
    "val_features": PROJECT_DIR / "data" / "splits_v2" / "val_features_v2.parquet",
    "test_features": PROJECT_DIR / "data" / "splits_v2" / "test_features_v2.parquet",
    "part_a_summary": PROJECT_DIR / "results" / "part_a_summary_v2.csv",
    "local_baseline_model": PROJECT_DIR / "outputs" / "runs_v2" / "xgboost__baseline__p001__20260404_143052" / "model.pkl",
}

print("\nLocal artefact checks:")
for name, path in local_paths.items():
    print(f"{name:24s}", path.exists(), path)

print("\nParquet metadata:")
for name in ["train", "val", "test", "train_features", "val_features", "test_features"]:
    path = local_paths[name]
    if path.exists():
        pf = pq.ParquetFile(path)
        print(f"{name:16s} rows={pf.metadata.num_rows:,} cols={pf.metadata.num_columns:,}")

print("\nSplit manifest:")
if local_paths["split_manifest"].exists():
    with open(local_paths["split_manifest"], "r") as f:
        manifest = json.load(f)
    print(json.dumps(manifest, indent=2)[:2000])

print("\nPart-A summary sanity check:")
if local_paths["part_a_summary"].exists():
    summary = pd.read_csv(local_paths["part_a_summary"])
    print("shape:", summary.shape)
    xgb_base = summary[
        (summary["model"].astype(str).str.lower() == "xgboost") &
        (summary["strategy"].astype(str).str.lower() == "baseline")
    ][["model", "strategy", "target_prevalence", "pr_auc_test", "roc_auc_test"]]
    display(xgb_base)

PART_A_RUN_DIR:
/content/drive/MyDrive/aml_results/large_run_v2_20260407_1904
exists: True

BASELINE_MODEL_PATH:
/content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl
exists: True
size MB: 0.8

Local artefact checks:
split_manifest           True /content/classimbalance/data/splits_v2/split_manifest.json
train                    True /content/classimbalance/data/splits_v2/train.parquet
val                      True /content/classimbalance/data/splits_v2/val.parquet
test                     True /content/classimbalance/data/splits_v2/test.parquet
train_features           True /content/classimbalance/data/splits_v2/train_features_v2.parquet
val_features             True /content/classimbalance/data/splits_v2/val_features_v2.parquet
test_features            True /content/classimbalance/data/splits_v2/test_features_v2.parquet
part_a_summary           True /content/classimbalance/results/part_a_summary_v2.csv
local_baseline_mod

,model,strategy,target_prevalence,pr_auc_test,roc_auc_test
0,xgboost,baseline,0.001,0.107854,0.966335
1,xgboost,baseline,0.005,0.107854,0.966335
2,xgboost,baseline,0.010,0.107854,0.966335


## 8 — Configure `paths_large_part_b_pai_hnu.yaml` for Colab


In [ ]:
import yaml, os
from pathlib import Path

os.chdir(PROJECT_DIR)

if not PATHS_CONFIG.exists():
    raise FileNotFoundError(f"Missing config: {PATHS_CONFIG}")

with open(PATHS_CONFIG, "r") as f:
    cfg = yaml.safe_load(f) or {}

# Keep raw_dir optional; Part B should use cached features/splits, not raw reprocessing.
cfg["raw_dir"] = str(DRIVE_ROOT / "aml_data")
cfg["processed_dir"] = str(PROJECT_DIR / "data" / "processed_v2")
cfg["splits_dir"] = str(PROJECT_DIR / "data" / "splits_v2")

# Part-B final outputs are isolated from Part-A outputs.
cfg["outputs_dir"] = str(PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu")
cfg["leaderboard_dir"] = str(PROJECT_DIR / "outputs" / "leaderboard_part_b_pai_hnu")

# Optional Part-A outputs only for reading/discovery. Never write Part-B results here.
cfg["part_a_outputs_dir"] = str(PROJECT_DIR / "outputs" / "runs_v2")

# Explicit model path can also be passed by CLI; keeping it here helps reproducibility.
cfg["baseline_model_path"] = str(BASELINE_MODEL_PATH)

with open(PATHS_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Updated config:")
print(yaml.safe_dump(cfg, sort_keys=False))


Updated config:
raw_dir: /content/drive/MyDrive/aml_data
processed_dir: /content/classimbalance/data/processed_v2
splits_dir: /content/classimbalance/data/splits_v2
outputs_dir: /content/classimbalance/outputs/runs_part_b_pai_hnu
leaderboard_dir: /content/classimbalance/outputs/leaderboard_part_b_pai_hnu
transactions_filename: LI-Large_Trans.csv
accounts_filename: LI-Large_accounts.csv
patterns_filename: LI-Large_Patterns.txt
output_transactions_labeled: transactions_labeled.parquet
split_manifest: split_manifest.json
part_a_summary: part_b_pai_hnu_summary.csv
baseline_model_path: /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl
part_a_outputs_dir: /content/classimbalance/outputs/runs_v2



## 9 — Sanity checks before scoring


In [ ]:
from pathlib import Path
import os, json

os.chdir(PROJECT_DIR)

print("Baseline model exists:", BASELINE_MODEL_PATH.exists(), BASELINE_MODEL_PATH)
if not BASELINE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Baseline model not found: {BASELINE_MODEL_PATH}")

# Required split/feature files are validated by project code too; this gives an early visual check.
splits_dir = PROJECT_DIR / "data" / "splits_v2"
print("\nFiles in data/splits_v2:")
for pattern in ["*train*", "*val*", "*test*", "*features*"]:
    matches = sorted(splits_dir.glob(pattern))
    print(f"\nPattern {pattern}: {len(matches)} matches")
    for f in matches[:20]:
        print(" -", f.name, f.stat().st_size / 1e6, "MB")

print("\nConfig path:", PATHS_CONFIG)


Baseline model exists: True /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl

Files in data/splits_v2:

Pattern *train*: 2 matches
 - train.parquet 3859.471611 MB
 - train_features_v2.parquet 7341.525082 MB

Pattern *val*: 2 matches
 - val.parquet 824.255305 MB
 - val_features_v2.parquet 1508.839888 MB

Pattern *test*: 2 matches
 - test.parquet 830.941973 MB
 - test_features_v2.parquet 1523.9732 MB

Pattern *features*: 3 matches
 - test_features_v2.parquet 1523.9732 MB
 - train_features_v2.parquet 7341.525082 MB
 - val_features_v2.parquet 1508.839888 MB

Config path: /content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml


## 10 — Level-3A: Generate/cache baseline train scores

This step uses the existing Part-A XGBoost Baseline model to score **training rows only**.  
Output should be `baseline_train_scores.parquet` with at least `row_idx` and `score`.




In [ ]:
import os, subprocess, sys
from pathlib import Path

os.chdir(PROJECT_DIR)

score_cache = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet"
score_meta = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json"

if score_cache.exists() and score_meta.exists():
    print("Score cache already exists. Skipping scoring.")
    print(" -", score_cache, score_cache.stat().st_size / 1e9, "GB")
    print(" -", score_meta)
else:
    cmd = [
        sys.executable, "-m", "aml_benchmark.experiments.score_baseline_train",
        "--paths", str(PATHS_CONFIG),
        "--baseline-model-path", str(BASELINE_MODEL_PATH),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("\nScore cache status:")
print("parquet exists:", score_cache.exists(), score_cache)
print("meta exists   :", score_meta.exists(), score_meta)


Running: /usr/bin/python3 -m aml_benchmark.experiments.score_baseline_train --paths /content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml --baseline-model-path /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl

Score cache status:
parquet exists: True /content/classimbalance/data/splits_v2/baseline_train_scores.parquet
meta exists   : True /content/classimbalance/data/splits_v2/baseline_train_scores_meta.json


In [ ]:
from pathlib import Path
import json
import pyarrow.parquet as pq
import pandas as pd

score_cache = Path("/content/classimbalance/data/splits_v2/baseline_train_scores.parquet")
score_meta = Path("/content/classimbalance/data/splits_v2/baseline_train_scores_meta.json")
train_features = Path("/content/classimbalance/data/splits_v2/train_features_v2.parquet")

print("Score cache exists:", score_cache.exists())
print("Score cache size GB:", round(score_cache.stat().st_size / 1e9, 3) if score_cache.exists() else None)
print("Meta exists:", score_meta.exists())
print("Meta size KB:", round(score_meta.stat().st_size / 1e3, 3) if score_meta.exists() else None)

print("\nParquet metadata:")
score_pf = pq.ParquetFile(score_cache)
train_pf = pq.ParquetFile(train_features)

print("score rows:", f"{score_pf.metadata.num_rows:,}")
print("score cols:", score_pf.schema.names)
print("train feature rows:", f"{train_pf.metadata.num_rows:,}")

print("\nRow count match:", score_pf.metadata.num_rows == train_pf.metadata.num_rows)

print("\nMeta JSON:")
with open(score_meta, "r") as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2)[:3000])

print("\nHead:")
display(pd.read_parquet(score_cache, columns=["row_idx", "score"]).head())

print("\nTail via last row group:")
last_rg = score_pf.num_row_groups - 1
tail_table = score_pf.read_row_group(last_rg, columns=["row_idx", "score"])
tail_df = tail_table.to_pandas().tail()
display(tail_df)

Score cache exists: True
Score cache size GB: 1.087
Meta exists: True
Meta size KB: 0.583

Parquet metadata:
score rows: 123,246,589
score cols: ['row_idx', 'score']
train feature rows: 123,246,589

Row count match: True

Meta JSON:
{
  "source_run_id": "xgboost__baseline__p001__20260404_143052",
  "model_path": "/content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl",
  "model_device_param": "cuda",
  "runtime_device": "cuda",
  "n_rows": 123246589,
  "score_min": 3.4199098308818066e-07,
  "score_max": 0.999534010887146,
  "score_mean": 0.0005182143650017679,
  "score_dtype": "float32",
  "scoring_time_sec": 41.75,
  "sha256_score_file": "08f970a063dadcb37fe6e529d2bba74d94ef2b5e00756fe3e69d4bbefb586bd4",
  "timestamp": "2026-05-01T12:31:50"
}

Head:


,row_idx,score
0,0,1.501066e-05
1,1,6.492545e-07
2,2,6.492545e-07
3,3,1.155431e-06
4,4,7.691185e-07



Tail via last row group:


,row_idx,score
563192,123246584,0.000003
563193,123246585,0.000321
563194,123246586,0.000096
563195,123246587,0.000003
563196,123246588,0.000175


## 11 — Inspect baseline score cache metadata


In [ ]:
import json
from pathlib import Path

score_meta = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json"
if score_meta.exists():
    meta = json.loads(score_meta.read_text())
    print(json.dumps(meta, indent=2)[:4000])
else:
    print("No meta file found.")

# Lightweight schema check, avoids loading all rows.
import pandas as pd
score_cache = PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet"
if score_cache.exists():
    df_head = pd.read_parquet(score_cache, columns=["row_idx", "score"]).head()
    print(df_head)
    print("Columns OK:", set(["row_idx", "score"]).issubset(df_head.columns))


{
  "source_run_id": "xgboost__baseline__p001__20260404_143052",
  "model_path": "/content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs/xgboost__baseline__p001__20260404_143052/model.pkl",
  "model_device_param": "cuda",
  "runtime_device": "cuda",
  "n_rows": 123246589,
  "score_min": 3.4199098308818066e-07,
  "score_max": 0.999534010887146,
  "score_mean": 0.0005182143650017679,
  "score_dtype": "float32",
  "scoring_time_sec": 41.75,
  "sha256_score_file": "08f970a063dadcb37fe6e529d2bba74d94ef2b5e00756fe3e69d4bbefb586bd4",
  "timestamp": "2026-05-01T12:31:50"
}
   row_idx         score
0        0  1.501066e-05
1        1  6.492545e-07
2        2  6.492545e-07
3        3  1.155431e-06
4        4  7.691185e-07
Columns OK: True


## 12 — Level-3B: Mini-End-to-End Smoke Run

This is the important smoke test. It trains on a deterministic row-aligned subset and writes to:

`outputs/runs_part_b_pai_hnu_smoke/<run_id>/`

Expected metadata:
- `smoke_subsample_used: true`
- `sample_n_train: 200000`
- `row_index_mode: internal_0_based_with_orig_row_idx_mapping`
- `subsample_row_mapping.parquet` exists

Do **not** run full Part-B runs before this is checked.


In [ ]:
import os, subprocess, sys
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, "-m", "aml_benchmark.experiments.run_part_b_pai_hnu",
    "--paths", str(PATHS_CONFIG),
    "--target-prevalences", SMOKE_PREVALENCE,
    "--sample-n-train", str(SMOKE_N_TRAIN),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: /usr/bin/python3 -m aml_benchmark.experiments.run_part_b_pai_hnu --paths /content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml --target-prevalences 0.01 --sample-n-train 200000


CompletedProcess(args=['/usr/bin/python3', '-m', 'aml_benchmark.experiments.run_part_b_pai_hnu', '--paths', '/content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml', '--target-prevalences', '0.01', '--sample-n-train', '200000'], returncode=0)

## 13 — Inspect latest Smoke output


In [ ]:
import json, os
from pathlib import Path
import pandas as pd

smoke_root = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu_smoke"
if not smoke_root.exists():
    raise FileNotFoundError(f"Smoke output root missing: {smoke_root}")

runs = sorted([p for p in smoke_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
if not runs:
    raise FileNotFoundError(f"No smoke run folders found in {smoke_root}")

latest = runs[0]
print("Latest smoke run:", latest)

def show_json(name, max_chars=5000):
    p = latest / name
    print("\n" + "=" * 80)
    print(name, "exists:", p.exists())
    if p.exists():
        data = json.loads(p.read_text())
        print(json.dumps(data, indent=2)[:max_chars])
    return p

run_config_p = show_json("run_config.json")
manifest_p = show_json("sampling_manifest.json")
show_json("metrics_val_opt.json")
show_json("metrics_test_opt.json")
show_json("metrics_val.json")
show_json("metrics_test.json")

mapping = latest / "subsample_row_mapping.parquet"
print("\n" + "=" * 80)
print("subsample_row_mapping.parquet exists:", mapping.exists())
if mapping.exists():
    m = pd.read_parquet(mapping)
    print(m.head())
    print("Rows:", len(m))
    print("Columns:", list(m.columns))
    print("orig_row_idx monotonic:", m["orig_row_idx"].is_monotonic_increasing)
    print("internal_row_idx starts at 0:", int(m["internal_row_idx"].iloc[0]) == 0)


Latest smoke run: /content/classimbalance/outputs/runs_part_b_pai_hnu_smoke/xgboost__pai_hnu__p010_SMOKE__20260501_123840

run_config.json exists: True
{
  "run_id": "xgboost__pai_hnu__p010_SMOKE__20260501_123840",
  "model": "xgboost",
  "strategy": "pai_hnu",
  "target_prevalence": 0.01,
  "achieved_train_prevalence": 0.01,
  "random_seed": 42,
  "is_smoke": true,
  "smoke_subsample_used": true,
  "sample_n_train": 200000,
  "row_index_mode": "internal_0_based_with_orig_row_idx_mapping",
  "subsample_row_mapping_parquet": "/content/classimbalance/outputs/runs_part_b_pai_hnu_smoke/xgboost__pai_hnu__p010_SMOKE__20260501_123840/subsample_row_mapping.parquet",
  "train_rows_after_sampling": 10200,
  "train_positives_after_sampling": 102,
  "train_negatives_after_sampling": 10098,
  "selection_counts": {
    "n_pos": 102,
    "n_neg_total_available": 199898,
    "n_neg_target": 10098,
    "n_hard_planned": 5049,
    "n_hard_cap": 2040,
    "n_hard_actual": 2040,
    "n_temporal_planned": 

In [ ]:
from pathlib import Path
import shutil
from datetime import datetime

PROJECT_DIR = Path("/content/classimbalance")
DRIVE_ROOT = Path("/content/drive/MyDrive/aml_results")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = DRIVE_ROOT / f"part_b_pai_hnu_progress_{timestamp}"
backup_dir.mkdir(parents=True, exist_ok=True)

print("Backup dir:", backup_dir)

# 1) Score cache sichern
score_files = [
    PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet",
    PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json",
]

for src in score_files:
    if src.exists():
        dst = backup_dir / src.name
        print(f"Copying {src} -> {dst}")
        shutil.copy2(src, dst)
    else:
        print("Missing:", src)

# 2) Smoke runs sichern
smoke_src = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu_smoke"
smoke_dst = backup_dir / "runs_part_b_pai_hnu_smoke"

if smoke_src.exists():
    print(f"Copying smoke outputs {smoke_src} -> {smoke_dst}")
    shutil.copytree(smoke_src, smoke_dst, dirs_exist_ok=True)
else:
    print("Smoke folder missing:", smoke_src)

# 3) Configs sichern
config_dst = backup_dir / "configs"
config_dst.mkdir(exist_ok=True)

for cfg in [
    PROJECT_DIR / "configs" / "paths_large_part_b_pai_hnu.yaml",
    PROJECT_DIR / "configs" / "benchmark_part_b_pai_hnu.yaml",
]:
    if cfg.exists():
        print(f"Copying config {cfg.name}")
        shutil.copy2(cfg, config_dst / cfg.name)
    else:
        print("Missing config:", cfg)

print("\nBackup complete.")
print("Saved to:", backup_dir)

Backup dir: /content/drive/MyDrive/aml_results/part_b_pai_hnu_progress_20260501_124452
Copying /content/classimbalance/data/splits_v2/baseline_train_scores.parquet -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_progress_20260501_124452/baseline_train_scores.parquet
Copying /content/classimbalance/data/splits_v2/baseline_train_scores_meta.json -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_progress_20260501_124452/baseline_train_scores_meta.json
Copying smoke outputs /content/classimbalance/outputs/runs_part_b_pai_hnu_smoke -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_progress_20260501_124452/runs_part_b_pai_hnu_smoke
Copying config paths_large_part_b_pai_hnu.yaml
Copying config benchmark_part_b_pai_hnu.yaml

Backup complete.
Saved to: /content/drive/MyDrive/aml_results/part_b_pai_hnu_progress_20260501_124452


## 14 — Back up Smoke output to Drive


In [ ]:
import shutil, datetime
from pathlib import Path

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_smoke_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

smoke_root = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu_smoke"
if smoke_root.exists():
    shutil.copytree(smoke_root, backup_dir / "runs_part_b_pai_hnu_smoke", dirs_exist_ok=True)
    print("Smoke outputs backed up to:", backup_dir)
else:
    print("Smoke root not found:", smoke_root)


Smoke outputs backed up to: /content/drive/MyDrive/aml_results/part_b_pai_hnu_smoke_20260501_1244


## 15 — Full Runs: manual confirmation guard


In [ ]:

RUN_FULL = True

if not RUN_FULL:
    raise RuntimeError("Full runs are blocked. Set RUN_FULL=True only after smoke output is approved.")


## 16 — Full Runs: separate run all three PAI-HNU prevalences

This cell runs the final three PAI-HNU experiments. It runs one prevalence at a time and backs up after each run.

Expected final output root:

`outputs/runs_part_b_pai_hnu/`


In [ ]:
import os, subprocess, sys, shutil, datetime
from pathlib import Path

os.chdir(PROJECT_DIR)

if not RUN_FULL:
    raise RuntimeError("RUN_FULL is False. Set RUN_FULL=True only after manual approval.")

p = "0.01"

print("\n" + "=" * 100)
print(f"Running full PAI-HNU prevalence {p}")
print("=" * 100)

cmd = [
    sys.executable, "-m", "aml_benchmark.experiments.run_part_b_pai_hnu",
    "--paths", str(PATHS_CONFIG),
    "--target-prevalences", p,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Backup after this prevalence
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_incremental_{p.replace('.', '')}_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

src = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu"
if src.exists():
    shutil.copytree(src, backup_dir / "runs_part_b_pai_hnu", dirs_exist_ok=True)
    print("Incremental backup:", backup_dir)
else:
    print("WARNING: Full output root not found:", src)


Running full PAI-HNU prevalence 0.01
Running: /usr/bin/python3 -m aml_benchmark.experiments.run_part_b_pai_hnu --paths /content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml --target-prevalences 0.01
Incremental backup: /content/drive/MyDrive/aml_results/part_b_pai_hnu_incremental_001_20260501_1254


Run p=0.005

In [ ]:
import os, subprocess, sys, shutil, datetime
from pathlib import Path

os.chdir(PROJECT_DIR)

if not RUN_FULL:
    raise RuntimeError("RUN_FULL is False. Set RUN_FULL=True only after manual approval.")

p = "0.005"

print("\n" + "=" * 100)
print(f"Running full PAI-HNU prevalence {p}")
print("=" * 100)

cmd = [
    sys.executable, "-m", "aml_benchmark.experiments.run_part_b_pai_hnu",
    "--paths", str(PATHS_CONFIG),
    "--target-prevalences", p,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_incremental_{p.replace('.', '')}_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

src = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu"
if src.exists():
    shutil.copytree(src, backup_dir / "runs_part_b_pai_hnu", dirs_exist_ok=True)
    print("Incremental backup:", backup_dir)
else:
    print("WARNING: Full output root not found:", src)


Running full PAI-HNU prevalence 0.005
Running: /usr/bin/python3 -m aml_benchmark.experiments.run_part_b_pai_hnu --paths /content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml --target-prevalences 0.005
Incremental backup: /content/drive/MyDrive/aml_results/part_b_pai_hnu_incremental_0005_20260501_1305


Run p=0.001

In [ ]:
import os, subprocess, sys, shutil, datetime
from pathlib import Path

os.chdir(PROJECT_DIR)

if not RUN_FULL:
    raise RuntimeError("RUN_FULL is False. Set RUN_FULL=True only after manual approval.")

p = "0.001"

print("\n" + "=" * 100)
print(f"Running full PAI-HNU prevalence {p}")
print("=" * 100)

cmd = [
    sys.executable, "-m", "aml_benchmark.experiments.run_part_b_pai_hnu",
    "--paths", str(PATHS_CONFIG),
    "--target-prevalences", p,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_incremental_{p.replace('.', '')}_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

src = PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu"
if src.exists():
    shutil.copytree(src, backup_dir / "runs_part_b_pai_hnu", dirs_exist_ok=True)
    print("Incremental backup:", backup_dir)
else:
    print("WARNING: Full output root not found:", src)


Running full PAI-HNU prevalence 0.001
Running: /usr/bin/python3 -m aml_benchmark.experiments.run_part_b_pai_hnu --paths /content/classimbalance/configs/paths_large_part_b_pai_hnu.yaml --target-prevalences 0.001
Incremental backup: /content/drive/MyDrive/aml_results/part_b_pai_hnu_incremental_0001_20260501_1317


In [ ]:
import json
from pathlib import Path

full_root = Path("/content/classimbalance/outputs/runs_part_b_pai_hnu")

runs = sorted(
    [p for p in full_root.iterdir() if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

latest = runs[0]
print("Latest full run:", latest)

def show_json(name, max_chars=6000):
    p = latest / name
    print("\n" + "=" * 80)
    print(name, "exists:", p.exists())
    if p.exists():
        data = json.loads(p.read_text())
        print(json.dumps(data, indent=2)[:max_chars])
    return p

for name in [
    "run_config.json",
    "sampling_manifest.json",
    "metrics_val_opt.json",
    "metrics_test_opt.json",
    "metrics_val.json",
    "metrics_test.json",
]:
    show_json(name)

Latest full run: /content/classimbalance/outputs/runs_part_b_pai_hnu/xgboost__pai_hnu__p001__20260501_131106

run_config.json exists: True
{
  "run_id": "xgboost__pai_hnu__p001__20260501_131106",
  "model": "xgboost",
  "strategy": "pai_hnu",
  "target_prevalence": 0.001,
  "achieved_train_prevalence": 0.001,
  "random_seed": 42,
  "is_smoke": false,
  "smoke_subsample_used": false,
  "sample_n_train": null,
  "row_index_mode": "full_train_row_order",
  "subsample_row_mapping_parquet": null,
  "train_rows_after_sampling": 63811000,
  "train_positives_after_sampling": 63811,
  "train_negatives_after_sampling": 63747189,
  "selection_counts": {
    "n_pos": 63811,
    "n_neg_total_available": 123182778,
    "n_neg_target": 63747189,
    "n_hard_planned": 31873594,
    "n_hard_cap": 1276220,
    "n_hard_actual": 1276220,
    "n_temporal_planned": 31235484,
    "n_temporal_actual": 31235484,
    "n_global_planned": 31235485,
    "n_global_actual": 31235485,
    "n_total_neg_actual": 637471

In [ ]:
from pathlib import Path
import hashlib
import json

full_root = Path("/content/classimbalance/outputs/runs_part_b_pai_hnu")

runs = sorted([p for p in full_root.iterdir() if p.is_dir()])

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for run in runs:
    cfg_path = run / "run_config.json"
    model_path = run / "model.pkl"
    print("\nRUN:", run.name)
    print("model exists:", model_path.exists())
    if model_path.exists():
        print("model size MB:", round(model_path.stat().st_size / 1e6, 3))
        print("model sha256:", sha256_file(model_path)[:16])
    if cfg_path.exists():
        cfg = json.loads(cfg_path.read_text())
        print("target_prevalence:", cfg.get("target_prevalence"))
        print("is_smoke:", cfg.get("is_smoke"))
        print("train_rows_after_sampling:", cfg.get("train_rows_after_sampling"))
        print("train_time_sec:", cfg.get("train_time_sec"))
        print("created_at:", cfg.get("created_at"))


RUN: xgboost__pai_hnu__p001__20260501_131106
model exists: True
model size MB: 0.793
model sha256: 12e00dccf9d671d5
target_prevalence: 0.001
is_smoke: False
train_rows_after_sampling: 63811000
train_time_sec: 65.69
created_at: 2026-05-01T13:17:11

RUN: xgboost__pai_hnu__p005__20260501_130210
model exists: True
model size MB: 0.786
model sha256: 783f80cfd47e0189
target_prevalence: 0.005
is_smoke: False
train_rows_after_sampling: 12762200
train_time_sec: 13.67
created_at: 2026-05-01T13:05:12

RUN: xgboost__pai_hnu__p010__20260501_125134
model exists: True
model size MB: 0.786
model sha256: 6526ebb56780b9f7
target_prevalence: 0.01
is_smoke: False
train_rows_after_sampling: 6381100
train_time_sec: 7.05
created_at: 2026-05-01T12:54:17


## 17 — Generate Table 6 after all three full runs


In [ ]:
import os, subprocess, sys
from pathlib import Path

os.chdir(PROJECT_DIR)

cmd = [sys.executable, "-m", "aml_benchmark.analysis.results_tables"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nCandidate result/table files:")
for root in [PROJECT_DIR / "results", PROJECT_DIR / "outputs", PROJECT_DIR / "outputs" / "leaderboard_part_b_pai_hnu"]:
    if root.exists():
        print("\nROOT:", root)
        for f in sorted(root.rglob("*table*"))[:50]:
            print(" -", f.relative_to(PROJECT_DIR))
        for f in sorted(root.rglob("*.csv"))[:50]:
            if "table" in f.name.lower() or "part" in f.name.lower():
                print(" -", f.relative_to(PROJECT_DIR))


Running: /usr/bin/python3 -m aml_benchmark.analysis.results_tables

Candidate result/table files:

ROOT: /content/classimbalance/results
 - results/tables
 - results/tables/table1a_main_results.csv
 - results/tables/table1a_main_results.md
 - results/tables/table1b_appendix_results.csv
 - results/tables/table1b_appendix_results.md
 - results/tables/table2_strategy6_comparison.csv
 - results/tables/table2_strategy6_comparison.md
 - results/tables/table3_feature_importance_xgboost.csv
 - results/tables/table3_feature_importance_xgboost.md
 - results/tables/table4_feature_importance_rf.csv
 - results/tables/table4_feature_importance_rf.md
 - results/tables/table6_pai_hnu_vs_part_a.csv
 - results/tables/table6_pai_hnu_vs_part_a.md
 - results/part_a_summary_v2.csv
 - results/tables/table1a_main_results.csv
 - results/tables/table1b_appendix_results.csv
 - results/tables/table2_strategy6_comparison.csv
 - results/tables/table3_feature_importance_xgboost.csv
 - results/tables/table4_feature_i

## 18 — Final Drive backup


In [ ]:
import shutil, datetime
from pathlib import Path

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
backup_dir = DRIVE_RESULTS / f"part_b_pai_hnu_run_{ts}"
backup_dir.mkdir(parents=True, exist_ok=True)

items = [
    PROJECT_DIR / "outputs" / "runs_part_b_pai_hnu",
    PROJECT_DIR / "outputs" / "leaderboard_part_b_pai_hnu",
    PROJECT_DIR / "results",
    PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores.parquet",
    PROJECT_DIR / "data" / "splits_v2" / "baseline_train_scores_meta.json",
]

for src in items:
    if not src.exists():
        print("Skip missing:", src)
        continue

    dst = backup_dir / src.relative_to(PROJECT_DIR)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Copied dir :", src.relative_to(PROJECT_DIR), "->", dst)
    else:
        shutil.copy2(src, dst)
        print("Copied file:", src.relative_to(PROJECT_DIR), "->", dst)

print("\nFinal backup complete:", backup_dir)


Copied dir : outputs/runs_part_b_pai_hnu -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/outputs/runs_part_b_pai_hnu
Skip missing: /content/classimbalance/outputs/leaderboard_part_b_pai_hnu
Copied dir : results -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/results
Copied file: data/splits_v2/baseline_train_scores.parquet -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/data/splits_v2/baseline_train_scores.parquet
Copied file: data/splits_v2/baseline_train_scores_meta.json -> /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/data/splits_v2/baseline_train_scores_meta.json

Final backup complete: /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322


## Notes

- `score_baseline_train.py` must only score training rows. Validation/test scores are not used for sampling.
- The smoke run uses internal `0..n_sub-1` indices after row-aligned subsampling and stores `orig_row_idx` in `subsample_row_mapping.parquet`.
- The full run operates on the full training row order.
- Table 6 intentionally requires all three full PAI-HNU prevalences. If one prevalence is missing, rerun only the missing prevalence.
- Back up after each full prevalence to avoid losing progress after Colab disconnects.


In [ ]:
from pathlib import Path
import json

drive_final = Path("/content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322")

print("Final Drive folder exists:", drive_final.exists())
print("Final Drive folder:", drive_final)

# Suche nach Run-Ordnern innerhalb des finalen Backups
run_dirs = sorted(drive_final.rglob("xgboost__pai_hnu__p*"))

print("\nFound run dirs:")
for p in run_dirs:
    if p.is_dir():
        print(" -", p)

print("\nRun configs:")
for run in run_dirs:
    cfg_path = run / "run_config.json"
    if cfg_path.exists():
        cfg = json.loads(cfg_path.read_text())
        print("\n", run.name)
        print(" target_prevalence:", cfg.get("target_prevalence"))
        print(" is_smoke:", cfg.get("is_smoke"))
        print(" train_rows_after_sampling:", cfg.get("train_rows_after_sampling"))
        print(" model exists:", (run / "model.pkl").exists())
        print(" metrics_test_opt exists:", (run / "metrics_test_opt.json").exists())

Final Drive folder exists: True
Final Drive folder: /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322

Found run dirs:
 - /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/outputs/runs_part_b_pai_hnu/xgboost__pai_hnu__p001__20260501_131106
 - /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/outputs/runs_part_b_pai_hnu/xgboost__pai_hnu__p005__20260501_130210
 - /content/drive/MyDrive/aml_results/part_b_pai_hnu_run_20260501_1322/outputs/runs_part_b_pai_hnu/xgboost__pai_hnu__p010__20260501_125134

Run configs:

 xgboost__pai_hnu__p001__20260501_131106
 target_prevalence: 0.001
 is_smoke: False
 train_rows_after_sampling: 63811000
 model exists: True
 metrics_test_opt exists: True

 xgboost__pai_hnu__p005__20260501_130210
 target_prevalence: 0.005
 is_smoke: False
 train_rows_after_sampling: 12762200
 model exists: True
 metrics_test_opt exists: True

 xgboost__pai_hnu__p010__20260501_125134
 target_prevalence: 0.01
 is_smoke: False
 t